# Track1039: fresh upgraded soft-voting ensemble

Attach the same compact PRIVATE **VGEC26-fast-quality-private-20261020.zip** dataset (2.17MB). Internet ON, **GPU T4 x2**. Repository-read GITHUB_TOKEN required; HF_TOKEN optional, WANDB_API_KEY enables monitoring. Save Version / Run All. No notebook toggles, old outputs or new data upload needed.

Train **Qwen2.5-14B + Sailor2-20B + Mistral-Nemo-12B classifiers**, five FULL epochs each, fresh from pinned pretrained bases. Retain the quality system's lambda2 soft hierarchy, class weighting and weighted SOFT voting1:1:2 (Qwen:Sailor:Mistral), temperature1, phrase scale1.25. Train a fresh **Qwen2.5-14B corrector for six FULL epochs**, select using development GLEU/clean preservation, then generate every private correction with greedy192 and no hints.

All models use NF4 double-quant QLoRA, FP16 compute, rank16/alpha32, seed42 and effective batch8. Classifiers micro4/eval8; corrector micro2, validation generation4, private generation8 with persistent OOM reduction8→4→2→1. Same quality data,78 development rows excluded from fitting,2034 private inputs inference-only. No folds, baseline comparisons, decoder sweep or saved-model score-replay passes.

Execution order is Sailor fit/private classification → Nemo fit/private classification → Qwen fit/private classification → soft vote → Qwen corrector fit/private correction → one results.zip. This keeps Qwen's downloaded base cached for correction. Model workers run sequentially across both T4s. Root W&B run shows live metrics, sanitized console and system resources throughout. No controller time cap; Kaggle's external limit still applies. Four fresh fits may take most of a session; full runtime, memory fit and quality are untested for this recipe.

Download **run/results.zip** at completion and retain the whole output with all three member adapters. This is a fresh experimental gamble, not a proven improvement over0.5806. The already prepared component-join candidate and original1038 quality archive remain intact. Details: agents/experiments/UPGRADED_ENSEMBLE_20261020.md.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "kaggle-linux"
SOURCE_ROOT = Path("/kaggle/working")
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Fetch the committed source securely into a fresh directory.
from urllib.parse import urlsplit
import tempfile

with tempfile.TemporaryDirectory() as authentication:
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    token = None
    if urlsplit(REPO_URL).hostname == "github.com":
        from kaggle_secrets import UserSecretsClient
        try:
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            raise RuntimeError("Enable GITHUB_TOKEN in Kaggle Secrets with repository read access.") from None
        if not token:
            raise RuntimeError("GITHUB_TOKEN is empty; add a repository-read token in Kaggle Secrets.")
        askpass = Path(authentication) / "git-askpass.sh"
        askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) printf "%s\\n" x-access-token ;;\n'
                           '*) printf "%s\\n" "$GECC_GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GECC_GITHUB_TOKEN=token)
    PROJECT_DIR = Path(tempfile.mkdtemp(prefix="gecc-source-", dir=SOURCE_ROOT))
    result = subprocess.run(["git", "-c", "credential.helper=", "clone", "--depth", "1",
                             "--single-branch", "--branch", BRANCH, "--", REPO_URL, str(PROJECT_DIR)],
                            env=env, capture_output=True, text=True)
    returncode = result.returncode
    detail = result.stderr.replace(token, "[REDACTED]") if token else result.stderr
    del env, token, result
    if returncode:
        raise RuntimeError("Cannot clone source; check Internet, branch, and GITHUB_TOKEN access. " + detail.strip())
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("gecc_notebook", PROJECT_DIR / "tools/notebook.py")
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)
TRAINING_PYTHON = launcher.setup_kaggle(PROJECT_DIR)


In [ ]:
RUN_DIR = launcher.launch(PROJECT_DIR, TRAINING_PYTHON)
